# Vietnamese Product Clustering

Notebook so sanh PhoBERT, E5 da ngon ngu va TF-IDF + SVD. Chay tu thu muc goc cua repository va theo thu tu cell. Cai dependency bang `requirements.txt` truoc khi chay.

In [ ]:

import os, sys, gc, math, pathlib, shutil
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_fp16 = torch.cuda.is_available()
dtype = torch.float16 if use_fp16 else torch.float32
print("Dtype dùng:", dtype)

OUT_DIR = "social_networking/pho_bert/outputs_phobert"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
import pandas as pd

CSV_PATH="description_phobert(1).csv"

df = pd.read_csv(CSV_PATH)

print("Cột:", df.columns.tolist())
print("Tổng dòng gốc:", len(df))

df["category_name"] = df["category_name"].fillna("unknown").astype(str)
df["description"]   = df["description"].fillna("").astype(str)

n_na        = df["description"].isna().sum()
n_empty     = (df["description"] == "").sum()
n_ws_only   = df["description"].str.strip().eq("").sum()

print(f"NaN: {n_na} | empty '': {n_empty} | whitespace-only: {n_ws_only}")

n_dups_desc = df.duplicated(subset=["description"]).sum()
print(f"Số dòng trùng mô tả (không tính dòng đầu tiên): {n_dups_desc}")

ex_ws = df[df["description"].str.strip().eq("")].head(5)
if len(ex_ws):
    print("\nVí dụ mô tả trắng/rỗng:")
    display(ex_ws)

df_keep = df[df["description"].str.strip().ne("")].reset_index(drop=True)
print("\nSố dòng sau khi loại trắng:", len(df_keep))


In [ ]:
df_keep["word_count"] = df_keep["description"].str.split().apply(len)
print(df_keep["word_count"].describe())


In [ ]:
total_words = df_keep["word_count"].sum()
print(f"Tổng số từ trong toàn bộ mô tả: {total_words:,}")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "vinai/phobert-large"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print("device:", device, "| dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device=device, dtype=dtype)
model.eval()



In [ ]:
import os
import py_vncorenlp

VNCORE_DIR = "social_networking/pho_bert/vncorenlp"
os.makedirs(VNCORE_DIR, exist_ok=True)

if "rdrsegmenter" not in globals():
    rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=VNCORE_DIR)
    print("VnCoreNLP loaded.")
else:
    print("Reusing existing rdrsegmenter (already loaded).")

def vn_word_segment(text: str) -> str:
    if not text.strip():
        return ""
    sents = rdrsegmenter.word_segment(text)
    return " ".join(sents)

df_keep["description_wseg"] = [vn_word_segment(t) for t in df_keep["description"]]

print("Ví dụ sau khi word-seg:")
print(df_keep[["description", "description_wseg"]].head(3))

df_keep.to_parquet("description_phobert_wseg.parquet", index=False)


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from contextlib import nullcontext

MAX_LEN = 256
BATCH_SIZE = 64

DATA_PATH = "social_networking/pho_bert/vncorenlp/description_phobert_wseg.parquet"
df_wseg = pd.read_parquet(DATA_PATH)
texts = df_wseg["description_wseg"].tolist()

class TextDataset(Dataset):
    def __init__(self, texts): self.texts = texts
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx]

dataset = TextDataset(texts)

def collate_batch(batch_texts):
    return tokenizer(
        batch_texts,
        padding="max_length",      
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_batch,
    num_workers=2 if device.type == "cuda" else 0,
    pin_memory=(device.type == "cuda")
)

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state      
    mask = attention_mask.unsqueeze(-1).expand_as(token_embeddings).float()
    return (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

all_embeddings = []
model.eval()

amp_ctx = torch.autocast(device_type="cuda", dtype=dtype) if device.type=="cuda" else nullcontext()

with torch.inference_mode():
    for batch in tqdm(loader, desc="Encoding"):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with amp_ctx:
            outputs = model(**batch)
            emb = mean_pooling(outputs, batch["attention_mask"])
            emb = torch.nn.functional.normalize(emb, p=2, dim=1) 
        all_embeddings.append(emb.cpu().numpy())

embeddings = np.vstack(all_embeddings)
assert embeddings.shape[0] == len(df_wseg), "Số embedding không khớp số dòng metadata!"

np.save("social_networking/pho_bert/outputs_phobert/phobert_embeddings.npy", embeddings)
df_wseg.to_parquet("social_networking/pho_bert/outputs_phobert/metadata_phobert.parquet", index=False)


In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

OUT_DIR   = "social_networking/pho_bert/outputs_phobert"
EMB_PATH  = f"{OUT_DIR}/phobert_embeddings.npy"
META_PATH = f"{OUT_DIR}/metadata_phobert.parquet"

assert Path(EMB_PATH).exists(), EMB_PATH
assert Path(META_PATH).exists(), META_PATH

embeddings = np.load(EMB_PATH)                 
meta       = pd.read_parquet(META_PATH)        
assert embeddings.shape[0] == len(meta)

print("Embeddings:", embeddings.shape)
print("Meta:", meta.shape)


In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score

EMB_PATH = "social_networking/pho_bert/outputs_phobert/phobert_embeddings.npy"
Ks = [4, 8, 12, 16, 20, 24, 28, 30, 40, 50]

X  = np.load(EMB_PATH)
Xn = normalize(X)

rows = []
for K in Ks:
    labels = KMeans(n_clusters=K, n_init="auto", random_state=42).fit_predict(Xn)
    sil = float(silhouette_score(Xn, labels, metric="cosine"))
    rows.append((K, sil))

df_k = pd.DataFrame(rows, columns=["K","silhouette_cosine"]).sort_values("silhouette_cosine", ascending=False)
best_K = int(df_k.iloc[0]["K"])
print(df_k.to_string(index=False))
print("best_K:", best_K)


In [ ]:
from sklearn.cluster import KMeans
import numpy as np

km = KMeans(n_clusters=best_K, n_init="auto", random_state=42).fit(Xn)
labels = km.labels_

uniq, cnt = np.unique(labels, return_counts=True)
print("best_K:", best_K, "| n_clusters:", len(uniq))
for k, c in zip(uniq, cnt):
    print(f"cluster {k}: {c}")


In [ ]:
import pandas as pd
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import os

EVAL_PATH = "social_networking/pho_bert/clustering/evaluation_phobert.csv"

sil = float(silhouette_score(Xn, labels, metric="cosine"))
dbi = float(davies_bouldin_score(Xn, labels))
ch  = float(calinski_harabasz_score(Xn, labels))

print(f"[Unsupervised] Silhouette(cosine)={sil:.4f} | DBI={dbi:.4f} | CH={ch:.2f}")

record = pd.DataFrame([{
    "model": "KMeans",
    "metric": "cosine",
    "K": best_K,
    "type": "unsupervised",
    "silhouette_cosine": sil,
    "DBI": dbi,
    "CH": ch,
    "NMI": None,
    "ARI": None,
    "Purity": None
}])

if os.path.exists(EVAL_PATH):
    old = pd.read_csv(EVAL_PATH)
    df_out = pd.concat([old, record], ignore_index=True)
else:
    df_out = record

df_out.to_csv(EVAL_PATH, index=False)


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
import os

META_PATH = "social_networking/pho_bert/outputs_phobert/metadata_phobert.parquet"
EVAL_PATH = "social_networking/pho_bert/clustering/evaluation_phobert.csv"

df_meta = pd.read_parquet(META_PATH)
y_true = LabelEncoder().fit_transform(df_meta["category_name"].astype(str).values)

def purity_score(y_true, y_pred):
    cm = contingency_matrix(y_true, y_pred)
    return cm.max(axis=0).sum() / cm.sum()

nmi = float(normalized_mutual_info_score(y_true, labels, average_method="arithmetic"))
ari = float(adjusted_rand_score(y_true, labels))
pur = float(purity_score(y_true, labels))

print(f"[Supervised] NMI={nmi:.4f} | ARI={ari:.4f} | Purity={pur:.4f}")

record = pd.DataFrame([{
    "model": "KMeans",
    "metric": "cosine",
    "K": best_K,
    "type": "supervised",
    "silhouette_cosine": None,
    "DBI": None,
    "CH": None,
    "NMI": nmi,
    "ARI": ari,
    "Purity": pur
}])

if os.path.exists(EVAL_PATH):
    old = pd.read_csv(EVAL_PATH)
    df_out = pd.concat([old, record], ignore_index=True)
else:
    df_out = record

df_out.to_csv(EVAL_PATH, index=False)


In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import normalize
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

EMB_PATH = "social_networking/pho_bert/outputs_phobert/phobert_embeddings.npy"
EPS_LIST = [0.003, 0.005, 0.007, 0.009, 0.011, 0.013]
MS_LIST  = [5, 8, 10, 12, 15]

X  = np.load(EMB_PATH)
Xn = normalize(X)

rows = []
for eps in EPS_LIST:
    for ms in MS_LIST:
        labels = DBSCAN(eps=eps, min_samples=ms, metric="cosine", n_jobs=-1).fit_predict(Xn)
        mask = labels != -1
        if mask.sum() >= 2 and len(np.unique(labels[mask])) >= 2:
            sil = float(silhouette_score(Xn[mask], labels[mask], metric="cosine"))
        else:
            sil = np.nan
        rows.append((eps, ms, sil, int((~mask).sum()) / len(labels)))

df = pd.DataFrame(rows, columns=["eps", "min_samples", "silhouette_cosine", "noise_frac"])
df = df.sort_values("silhouette_cosine", ascending=False, na_position="last")

print(df.head(15).to_string(index=False))

best = df.dropna(subset=["silhouette_cosine"]).iloc[0]
best_eps = float(best.eps)
best_ms  = int(best.min_samples)

print(f"\nChọn: eps={best_eps}, min_samples={best_ms} | sil={best.silhouette_cosine:.4f}")

labels_best = DBSCAN(eps=best_eps, min_samples=best_ms, metric="cosine", n_jobs=-1).fit_predict(Xn)
uniq, cnt = np.unique(labels_best[labels_best != -1], return_counts=True)
noise = int((labels_best == -1).sum())
print(f"clusters={len(uniq)} | noise={noise}/{len(labels_best)} ({noise/len(labels_best):.2%})")
for k, c in zip(uniq, cnt):
    print(f"cluster {k}: {c}")


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize
import numpy as np


db = DBSCAN(eps=0.007, min_samples=15, metric="cosine", n_jobs=-1).fit(Xn)
labels = db.labels_

noise = int((labels == -1).sum())
uniq, cnt = np.unique(labels[labels != -1], return_counts=True)
print(f"eps={0.007}, min_samples={15}")
print(f"clusters={len(uniq)} | noise={noise} / {len(labels)} ({noise/len(labels):.2%})")
for k, c in zip(uniq, cnt):
    print(f"cluster {k}: {c}")


In [ ]:
import pandas as pd, os
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

EVAL_PATH = "social_networking/pho_bert/clustering/evaluation_phobert.csv"

mask = (labels != -1)
if mask.sum() >= 2 and len(np.unique(labels[mask])) >= 2:
    sil = float(silhouette_score(Xn[mask], labels[mask], metric="cosine"))
    dbi = float(davies_bouldin_score(Xn[mask], labels[mask]))
    ch  = float(calinski_harabasz_score(Xn[mask], labels[mask]))
else:
    sil = dbi = ch = float("nan")

print(f"[DBSCAN-cosine | unsupervised] sil={sil:.4f} | DBI={dbi} | CH={ch}")

record = pd.DataFrame([{
    "model": "DBSCAN",
    "metric": "cosine",
    "K": "",            
    "type": "unsupervised",
    "silhouette_cosine": sil,
    "DBI": dbi,
    "CH": ch,
    "NMI": None,
    "ARI": None,
    "Purity": None
}])

if os.path.exists(EVAL_PATH):
    old = pd.read_csv(EVAL_PATH)
    df_out = pd.concat([old, record], ignore_index=True)
else:
    df_out = record

df_out.to_csv(EVAL_PATH, index=False)



In [ ]:
import pandas as pd, os
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix

META_PATH = "social_networking/pho_bert/outputs_phobert/metadata_phobert.parquet"
EVAL_PATH = "social_networking/pho_bert/clustering/evaluation_phobert.csv"

df_meta = pd.read_parquet(META_PATH)
y_true = LabelEncoder().fit_transform(df_meta["category_name"].astype(str).values)

def purity_score(y_true, y_pred):
    cm = contingency_matrix(y_true, y_pred)
    return cm.max(axis=0).sum() / cm.sum()

nmi = float(normalized_mutual_info_score(y_true, labels, average_method="arithmetic"))
ari = float(adjusted_rand_score(y_true, labels))
pur = float(purity_score(y_true, labels))

print(f"[DBSCAN-cosine | supervised] NMI={nmi:.4f} | ARI={ari:.4f} | Purity={pur:.4f}")

record = pd.DataFrame([{
    "model": "DBSCAN",
    "metric": "cosine",
    "K": "",
    "type": "supervised",
    "silhouette_cosine": None,
    "DBI": None,
    "CH": None,
    "NMI": nmi,
    "ARI": ari,
    "Purity": pur
}])

if os.path.exists(EVAL_PATH):
    old = pd.read_csv(EVAL_PATH)
    df_out = pd.concat([old, record], ignore_index=True)
else:
    df_out = record

df_out.to_csv(EVAL_PATH, index=False)


In [ ]:
import numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score

EMB_PATH = "social_networking/pho_bert/outputs_phobert/phobert_embeddings.npy"
Ks = [4, 8, 12, 14, 16, 20, 24, 28, 30]

X = np.load(EMB_PATH).astype(np.float64)
Xn = normalize(X)

rows = []
for K in Ks:
    gmm = GaussianMixture(n_components=K, covariance_type="diag", reg_covar=1e-4, random_state=42)
    labels = gmm.fit_predict(Xn)
    sil = float(silhouette_score(Xn, labels, metric="cosine"))
    rows.append((K, sil))

df_k = pd.DataFrame(rows, columns=["K", "silhouette_cosine"]).sort_values("silhouette_cosine", ascending=False)
best_K = int(df_k.iloc[0]["K"])
print(df_k.to_string(index=False))
print("best_K:", best_K)



In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np

gmm = GaussianMixture(n_components=best_K, covariance_type="diag", reg_covar=1e-4, random_state=42)
labels = gmm.fit_predict(Xn)

uniq, cnt = np.unique(labels, return_counts=True)
print("best_K:", best_K, "| n_clusters:", len(uniq))
for k, c in zip(uniq, cnt):
    print(f"cluster {k}: {c}")



In [ ]:
import pandas as pd, os
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

EVAL_PATH = "social_networking/pho_bert/clustering/evaluation_phobert.csv"

sil = float(silhouette_score(Xn, labels, metric="cosine"))
dbi = float(davies_bouldin_score(Xn, labels))
ch = float(calinski_harabasz_score(Xn, labels))

print(f"[Unsupervised] Silhouette(cosine)={sil:.4f} | DBI={dbi:.4f} | CH={ch:.2f}")

record = pd.DataFrame([{
    "model": "GMM",
    "metric": "cosine",
    "K": best_K,
    "type": "unsupervised",
    "silhouette_cosine": sil,
    "DBI": dbi,
    "CH": ch,
    "NMI": None,
    "ARI": None,
    "Purity": None
}])

if os.path.exists(EVAL_PATH):
    old = pd.read_csv(EVAL_PATH)
    df_out = pd.concat([old, record], ignore_index=True)
else:
    df_out = record

df_out.to_csv(EVAL_PATH, index=False)



In [ ]:
import pandas as pd, os
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix

META_PATH = "social_networking/pho_bert/outputs_phobert/metadata_phobert.parquet"
EVAL_PATH = "social_networking/pho_bert/clustering/evaluation_phobert.csv"

df_meta = pd.read_parquet(META_PATH)
y_true = LabelEncoder().fit_transform(df_meta["category_name"].astype(str).values)

def purity_score(y_true, y_pred):
    cm = contingency_matrix(y_true, y_pred)
    return cm.max(axis=0).sum() / cm.sum()

nmi = float(normalized_mutual_info_score(y_true, labels, average_method="arithmetic"))
ari = float(adjusted_rand_score(y_true, labels))
pur = float(purity_score(y_true, labels))

print(f"[Supervised] NMI={nmi:.4f} | ARI={ari:.4f} | Purity={pur:.4f}")

record = pd.DataFrame([{
    "model": "GMM",
    "metric": "cosine",
    "K": best_K,
    "type": "supervised",
    "silhouette_cosine": None,
    "DBI": None,
    "CH": None,
    "NMI": nmi,
    "ARI": ari,
    "Purity": pur
}])

if os.path.exists(EVAL_PATH):
    old = pd.read_csv(EVAL_PATH)
    df_out = pd.concat([old, record], ignore_index=True)
else:
    df_out = record

df_out.to_csv(EVAL_PATH, index=False)



In [ ]:
import pandas as pd

CSV_PATH = "social_networking/multilingual/description_multilingual.csv"

df = pd.read_csv(CSV_PATH)
print("Cột:", df.columns.tolist())
print("Tổng dòng gốc:", len(df))

df["category_name"] = df["category_name"].fillna("unknown").astype(str)
df["description"]   = df["description"].fillna("").astype(str)

df_keep = df[df["description"].str.strip().ne("")].reset_index(drop=True)
print("Số dòng sau khi loại trắng:", len(df_keep))

df_keep["word_count"] = df_keep["description"].str.split().apply(len)

print("\nThống kê độ dài mô tả:")
print(df_keep["word_count"].describe())

total_words = df_keep["word_count"].sum()
print(f"\nTổng số từ trong toàn bộ mô tả: {total_words:,}")


In [ ]:
import os, pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer

CSV_PATH = "social_networking/multilingual/description_multilingual.csv"
OUT_DIR  = "social_networking/multilingual/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("intfloat/multilingual-e5-large")

df = pd.read_csv(CSV_PATH)
df["description"] = df["description"].fillna("").astype(str)
df = df[df["description"].str.strip().ne("")].reset_index(drop=True)

texts = ["passage: " + t for t in df["description"].tolist()]

emb = model.encode(
    texts,
    batch_size=16,                 
    convert_to_numpy=True,
    normalize_embeddings=True,    
    show_progress_bar=True
)

np.save(f"{OUT_DIR}/embeddings.npy", emb)
df.to_parquet(f"{OUT_DIR}/metadata.parquet", index=False)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

EMB_PATH  = "social_networking/multilingual/outputs/embeddings.npy"
META_PATH = "social_networking/multilingual/outputs/metadata.parquet"

Xn   = np.load(EMB_PATH)
meta = pd.read_parquet(META_PATH)

Ks = [4, 8, 12, 16, 20, 24, 28, 30, 40, 50]
rows = []

for K in Ks:
    print(f"Đang chạy K={K} ...")
    kmeans = KMeans(n_clusters=K, n_init="auto", random_state=42)
    labels = kmeans.fit_predict(Xn)
    sil = float(silhouette_score(Xn, labels, metric="cosine"))
    rows.append((K, sil))

df_k = pd.DataFrame(rows, columns=["K", "silhouette_cosine"]).sort_values(
    "silhouette_cosine", ascending=False
)
best_K = int(df_k.iloc[0]["K"])

print("\n=== Kết quả Grid Search ===")
print(df_k.to_string(index=False))
print(f"\nBest K = {best_K} (theo silhouette_cosine cao nhất)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import os

EMB_PATH  = "social_networking/multilingual/outputs/embeddings.npy"
META_PATH = "social_networking/multilingual/outputs/metadata.parquet"
OUT_PATH  = "social_networking/multilingual/clustering/clusters.parquet"

Xn   = np.load(EMB_PATH)
meta = pd.read_parquet(META_PATH)

kmeans = KMeans(n_clusters=8, n_init="auto", random_state=42)
labels = kmeans.fit_predict(Xn)
meta["cluster_id"] = labels

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
meta.to_parquet(OUT_PATH, index=False)

print("Số cụm dự đoán (cluster_id):", meta["cluster_id"].nunique())
print(meta["cluster_id"].value_counts().sort_index())

if "category_name" in meta.columns:
    mask = meta["category_name"].notna()
    n_true = meta.loc[mask, "category_name"].nunique()
    print("\nSố cụm thật (category_name):", n_true)
    print(meta.loc[mask, "category_name"].value_counts().head(20))
else:
    print("\nKhông có cột 'category_name' trong metadata.")


In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score,
    homogeneity_score, completeness_score, v_measure_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.cluster import contingency_matrix

EVAL_PATH = "social_networking/multilingual/clustering/evaluation_multilingual.csv"
best_K = 8 

sil = float(silhouette_score(Xn, labels, metric="cosine"))
dbi = float(davies_bouldin_score(Xn, labels))
ch  = float(calinski_harabasz_score(Xn, labels))
print(f"[Unsupervised] Silhouette={sil:.4f} | DBI={dbi:.4f} | CH={ch:.2f}")

nmi = ari = pur = homo = comp = vmes = None
if "category_name" in meta.columns:
    mask = meta["category_name"].notna()
    if mask.any():
        y_true = LabelEncoder().fit_transform(meta.loc[mask, "category_name"].astype(str).values)
        y_pred = meta.loc[mask, "cluster_id"].values

        def purity_score(y_true, y_pred):
            cm = contingency_matrix(y_true, y_pred)
            return cm.max(axis=0).sum() / cm.sum()

        nmi  = float(normalized_mutual_info_score(y_true, y_pred, average_method="arithmetic"))
        ari  = float(adjusted_rand_score(y_true, y_pred))
        pur  = float(purity_score(y_true, y_pred))
        homo = float(homogeneity_score(y_true, y_pred))
        comp = float(completeness_score(y_true, y_pred))
        vmes = float(v_measure_score(y_true, y_pred))

        print(f"[Supervised] NMI={nmi:.4f} | ARI={ari:.4f} | Purity={pur:.4f} | Homogeneity={homo:.4f} | Completeness={comp:.4f} | V={vmes:.4f}")
    else:
        print("[Supervised] Không có bản ghi hợp lệ trong category_name")
else:
    print("[Supervised] Không có cột category_name")

rec_unsup = pd.DataFrame([{
    "model": "KMeans", "metric": "cosine", "K": best_K, "type": "unsupervised",
    "silhouette_cosine": sil, "DBI": dbi, "CH": ch,
    "NMI": None, "ARI": None, "Purity": None,
    "Homogeneity": None, "Completeness": None, "V_measure": None
}])

rec_sup = pd.DataFrame([{
    "model": "KMeans", "metric": "cosine", "K": best_K, "type": "supervised",
    "silhouette_cosine": None, "DBI": None, "CH": None,
    "NMI": nmi, "ARI": ari, "Purity": pur,
    "Homogeneity": homo, "Completeness": comp, "V_measure": vmes
}])

df_out = pd.read_csv(EVAL_PATH) if os.path.exists(EVAL_PATH) else pd.DataFrame()
df_out = pd.concat([df_out, rec_unsup, rec_sup], ignore_index=True)
df_out.to_csv(EVAL_PATH, index=False)


In [ ]:
pd.crosstab(meta["category_name"], meta["cluster_id"])


In [ ]:
from collections import Counter
for cid, group in meta.groupby("cluster_id"):
    top = group["category_name"].value_counts(normalize=True).head(3)
    print(f"Cluster {cid}: {dict(top)}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ct = pd.crosstab(meta["category_name"], meta["cluster_id"], normalize='index')
plt.figure(figsize=(10,6))
sns.heatmap(ct, cmap="YlGnBu")
plt.title("Phân bố category_name trong từng cluster_id")
plt.show()


In [ ]:
EMB_PATH  = "social_networking/multilingual/outputs/embeddings.npy"
META_PATH = "social_networking/multilingual/outputs/metadata.parquet"



In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

EMB_PATH  = "social_networking/multilingual/outputs/embeddings.npy"
META_PATH = "social_networking/multilingual/outputs/metadata.parquet"

Xn   = np.load(EMB_PATH)
meta = pd.read_parquet(META_PATH)

eps_list = np.round(np.arange(0.03, 0.13, 0.02), 3) 
min_samples_list = [5, 10, 20]

rows = []
for eps in eps_list:
    for ms in min_samples_list:
        db = DBSCAN(eps=eps, min_samples=ms, metric="cosine", n_jobs=-1)
        labels_tmp = db.fit_predict(Xn)
        mask = labels_tmp != -1
        n_valid = mask.sum()
        n_clusters = len(np.unique(labels_tmp[mask])) if n_valid > 0 else 0
        noise_ratio = 1.0 - (n_valid / len(labels_tmp))
        if n_clusters >= 2:
            sil = float(silhouette_score(Xn[mask], labels_tmp[mask], metric="cosine"))
            dbi = float(davies_bouldin_score(Xn[mask], labels_tmp[mask]))
            ch  = float(calinski_harabasz_score(Xn[mask], labels_tmp[mask]))
        else:
            sil = np.nan; dbi = np.nan; ch = np.nan
        rows.append({"eps": eps, "min_samples": ms, "n_clusters": n_clusters,
                     "noise_ratio": noise_ratio, "silhouette_cosine": sil, "DBI": dbi, "CH": ch})

df_gs = pd.DataFrame(rows)
df_gs_sorted = df_gs.copy()
df_gs_sorted["sil_rank"] = df_gs_sorted["silhouette_cosine"].fillna(-1e9)
df_gs_sorted = df_gs_sorted.sort_values(
    by=["sil_rank", "n_clusters", "noise_ratio"],
    ascending=[False, False, True]
).drop(columns=["sil_rank"])

best_eps = float(df_gs_sorted.iloc[0]["eps"])
best_ms  = int(df_gs_sorted.iloc[0]["min_samples"])

print("Top 10:")
print(df_gs_sorted.head(10).to_string(index=False))
print(f"\nBest params: eps={best_eps}, min_samples={best_ms}")


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
import pandas as pd
import os

OUT_PATH_DBSCAN = "social_networking/multilingual/clustering/clusters_DBSCAN.parquet"

db_best = DBSCAN(eps=best_eps, min_samples=best_ms, metric="cosine", n_jobs=-1)
labels_dbscan = db_best.fit_predict(Xn)
meta["cluster_id_dbscan"] = labels_dbscan

os.makedirs(os.path.dirname(OUT_PATH_DBSCAN), exist_ok=True)
meta.to_parquet(OUT_PATH_DBSCAN, index=False)

mask = labels_dbscan != -1
n_clusters = len(np.unique(labels_dbscan[mask])) if mask.any() else 0
print("Số cụm (không tính noise):", n_clusters)
print(pd.Series(labels_dbscan[mask]).value_counts().sort_index())


In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.cluster import contingency_matrix

EVAL_PATH = "social_networking/multilingual/clustering/evaluation_multilingual.csv"

mask = meta["cluster_id_dbscan"].values != -1
if mask.sum() >= 2 and len(np.unique(meta.loc[mask, "cluster_id_dbscan"])) >= 2:
    sil = float(silhouette_score(Xn[mask], meta.loc[mask, "cluster_id_dbscan"], metric="cosine"))
    dbi = float(davies_bouldin_score(Xn[mask], meta.loc[mask, "cluster_id_dbscan"]))
    ch  = float(calinski_harabasz_score(Xn[mask], meta.loc[mask, "cluster_id_dbscan"]))
else:
    sil = np.nan; dbi = np.nan; ch = np.nan
print(f"[DBSCAN | Unsupervised] Silhouette={sil} | DBI={dbi} | CH={ch}")

nmi = ari = pur = None
if "category_name" in meta.columns:
    mask2 = mask & meta["category_name"].notna()
    if mask2.any() and len(np.unique(meta.loc[mask2, "cluster_id_dbscan"])) >= 2:
        y_true = LabelEncoder().fit_transform(meta.loc[mask2, "category_name"].astype(str))
        y_pred = meta.loc[mask2, "cluster_id_dbscan"].values

        def purity_score(y_true, y_pred):
            cm = contingency_matrix(y_true, y_pred)
            return cm.max(axis=0).sum() / cm.sum()

        nmi = float(normalized_mutual_info_score(y_true, y_pred, average_method="arithmetic"))
        ari = float(adjusted_rand_score(y_true, y_pred))
        pur = float(purity_score(y_true, y_pred))
        print(f"[DBSCAN | Supervised] NMI={nmi} | ARI={ari} | Purity={pur}")

rec_unsup = pd.DataFrame([{
    "model": "DBSCAN", "metric": "cosine",
    "eps": best_eps, "min_samples": best_ms,
    "type": "unsupervised",
    "silhouette_cosine": sil, "DBI": dbi, "CH": ch,
    "NMI": None, "ARI": None, "Purity": None
}])

rec_sup = pd.DataFrame([{
    "model": "DBSCAN", "metric": "cosine",
    "eps": best_eps, "min_samples": best_ms,
    "type": "supervised",
    "silhouette_cosine": None, "DBI": None, "CH": None,
    "NMI": nmi, "ARI": ari, "Purity": pur
}])

df_out = pd.read_csv(EVAL_PATH) if os.path.exists(EVAL_PATH) else pd.DataFrame()
df_out = pd.concat([df_out, rec_unsup, rec_sup], ignore_index=True)
df_out.to_csv(EVAL_PATH, index=False)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_vis = meta.copy()
df_vis = df_vis[df_vis["cluster_id_dbscan"] != -1]
if "category_name" in df_vis.columns:
    ct = pd.crosstab(df_vis["category_name"], df_vis["cluster_id_dbscan"], normalize="index")
    plt.figure(figsize=(10, 6))
    sns.heatmap(ct, cmap="YlGnBu")
    plt.title("Phân bố category_name theo cluster_id_dbscan (DBSCAN, bỏ noise)")
    plt.tight_layout()
    plt.show()
else:
    print("Không có cột category_name để trực quan.")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

Xn = np.load(EMB_PATH)
meta = pd.read_parquet(META_PATH)

n_components_list = [6, 10, 14, 16, 20, 30]
rows = []

for k in n_components_list:
    try:
        gmm = GaussianMixture(
            n_components=k,
            covariance_type="diag", 
            reg_covar=1e-4,
            random_state=42
        ).fit(Xn)
        labels_tmp = gmm.predict(Xn)

        sil = float(silhouette_score(Xn, labels_tmp, metric="cosine"))
        rows.append({"n_components": k, "silhouette_cosine": sil})
    except Exception:
        rows.append({"n_components": k, "silhouette_cosine": np.nan})

df_gmm = pd.DataFrame(rows).sort_values(
    "silhouette_cosine", ascending=False
).reset_index(drop=True)

best_n = int(df_gmm.iloc[0]["n_components"])

print(df_gmm.to_string(index=False))
print(f"\nBest params (by silhouette): n_components={best_n}")


In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np
import pandas as pd
import os

OUT_PATH_GMM = "social_networking/multilingual/clustering/clusters_GMM.parquet"

gmm = GaussianMixture(
    n_components=best_n,
    covariance_type="diag",  
    reg_covar=1e-4,
    random_state=42
)
labels_gmm = gmm.fit_predict(Xn)
meta["cluster_id_gmm"] = labels_gmm

os.makedirs(os.path.dirname(OUT_PATH_GMM), exist_ok=True)
meta.to_parquet(OUT_PATH_GMM, index=False)

uniq, cnt = np.unique(labels_gmm, return_counts=True)
print("best_K:", best_n, "| n_clusters:", len(uniq))
for k, c in zip(uniq, cnt):
    print(f"cluster {k}: {c}")

if "category_name" in meta.columns:
    n_true = meta["category_name"].nunique()
    print("\nSố cụm thật (category_name):", n_true)


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.cluster import contingency_matrix

EVAL_PATH = "social_networking/multilingual/clustering/evaluation_multilingual.csv"

n_clusters = np.unique(labels_gmm).size
if n_clusters >= 2:
    sil = float(silhouette_score(Xn, labels_gmm, metric="cosine"))
    dbi = float(davies_bouldin_score(Xn, labels_gmm))
    ch  = float(calinski_harabasz_score(Xn, labels_gmm))
else:
    sil = dbi = ch = np.nan
print(f"[GMM | Unsupervised] Silhouette={sil:.4f} | DBI={dbi:.4f} | CH={ch:.2f}")

nmi = ari = pur = None
if "category_name" in meta.columns:
    mask = meta["category_name"].notna()
    if mask.any() and meta.loc[mask, "cluster_id_gmm"].nunique() >= 2:
        y_true = LabelEncoder().fit_transform(meta.loc[mask, "category_name"].astype(str))
        y_pred = meta.loc[mask, "cluster_id_gmm"].values

        def purity_score(y_true, y_pred):
            cm = contingency_matrix(y_true, y_pred)
            return cm.max(axis=0).sum() / cm.sum()

        nmi = float(normalized_mutual_info_score(y_true, y_pred, average_method="arithmetic"))
        ari = float(adjusted_rand_score(y_true, y_pred))
        pur = float(purity_score(y_true, y_pred))
        print(f"[GMM | Supervised] NMI={nmi:.4f} | ARI={ari:.4f} | Purity={pur:.4f}")

rec_unsup = pd.DataFrame([{
    "model": "GMM",
    "n_components": best_n,
    "type": "unsupervised",
    "silhouette_metric": "cosine",
    "silhouette_cosine": sil,
    "DBI": dbi,
    "CH": ch,
    "NMI": None,
    "ARI": None,
    "Purity": None
}])

rec_sup = pd.DataFrame([{
    "model": "GMM",
    "n_components": best_n,
    "type": "supervised",
    "silhouette_metric": None,
    "silhouette_cosine": None,
    "DBI": None,
    "CH": None,
    "NMI": nmi,
    "ARI": ari,
    "Purity": pur
}])

df_out = pd.read_csv(EVAL_PATH) if os.path.exists(EVAL_PATH) else pd.DataFrame()
df_out = pd.concat([df_out, rec_unsup, rec_sup], ignore_index=True)
df_out.to_csv(EVAL_PATH, index=False)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df_viz = (
    meta.groupby(["category_name", "cluster_id_gmm"])
        .size()
        .reset_index(name="count")
)

df_viz["ratio"] = (
    df_viz.groupby("category_name")["count"].transform(lambda x: x / x.sum())
)

heatmap_data = df_viz.pivot(index="category_name", columns="cluster_id_gmm", values="ratio").fillna(0)

plt.figure(figsize=(12, 7))
sns.heatmap(
    heatmap_data,
    cmap="YlGnBu",
    linewidths=0.5,
    annot=False,
    cbar_kws={'label': 'Tỷ lệ mẫu trong cụm'}
)
plt.title("Phân bố category_name theo cluster_id_gmm (GMM)", fontsize=14)
plt.xlabel("cluster_id_gmm")
plt.ylabel("category_name")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

CSV_PATH = "social_networking/traditional/description_clean.csv"

df = pd.read_csv(CSV_PATH)
print("Cột:", df.columns.tolist())
print("Tổng dòng gốc:", len(df))

df["category_name"] = df["category_name"].fillna("unknown").astype(str)
df["description"]   = df["description"].fillna("").astype(str)

df_keep = df[df["description"].str.strip().ne("")].reset_index(drop=True)
print("Số dòng sau khi loại trắng:", len(df_keep))

df_keep["word_count"] = df_keep["description"].str.split().apply(len)

print("\nThống kê độ dài mô tả:")
print(df_keep["word_count"].describe())

total_words = df_keep["word_count"].sum()
print(f"\nTổng số từ trong toàn bộ mô tả: {total_words:,}")


In [ ]:
import pandas as pd

CSV_PATH = "social_networking/traditional/description_clean.csv"

df = pd.read_csv(CSV_PATH)

print("Cột:", df.columns.tolist())
print("Tổng dòng gốc:", len(df))

required_cols = {"category_name", "description"}
missing = required_cols - set(df.columns)
assert not missing, f"Thiếu cột: {missing}"

df["category_name"] = df["category_name"].fillna("unknown").astype(str)
df["description"]   = df["description"].fillna("").astype(str)

n_na      = df["description"].isna().sum()
n_empty   = (df["description"] == "").sum()
n_ws_only = df["description"].str.strip().eq("").sum()
print(f"NaN: {n_na} | empty '': {n_empty} | whitespace-only: {n_ws_only}")

n_dups_desc = df.duplicated(subset=["description"]).sum()
print(f"Số dòng trùng mô tả (không tính dòng đầu tiên): {n_dups_desc}")

df_keep = df[df["description"].str.strip().ne("")].reset_index(drop=True)
print("Số dòng sau khi loại trắng:", len(df_keep))

print("\nVí dụ mô tả:")
print(df_keep[["category_name", "description"]].head(3))


In [ ]:
import numpy as np
import pandas as pd
import os
from scipy import sparse
from joblib import dump
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

OUT_DIR = "social_networking/traditional/outputs_traditional"
os.makedirs(OUT_DIR, exist_ok=True)

texts = df_keep["description"].tolist()

tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    dtype=np.float32,
    norm="l2"
)
X_tfidf = tfidf.fit_transform(texts)
print("TF-IDF shape:", X_tfidf.shape)
density = X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])
print(f"Sparse density: {density:.6f} ({density*100:.4f}%)")

SPARSE_PATH = f"{OUT_DIR}/tfidf_word.npz"
sparse.save_npz(SPARSE_PATH, X_tfidf)
print("Đã lưu TF-IDF sparse:", SPARSE_PATH)

SVD_DIM = 256
svd = TruncatedSVD(n_components=SVD_DIM, random_state=42, n_iter=7) 
X_svd = svd.fit_transform(X_tfidf)
Xn = normalize(X_svd)
print("SVD embedding shape:", Xn.shape)

SVD_PATH = f"{OUT_DIR}/traditional_embeddings.npy"
np.save(SVD_PATH, Xn.astype(np.float32))
print("Đã lưu embedding:", SVD_PATH)

META_PATH = f"{OUT_DIR}/metadata_traditional.parquet"
df_keep.to_parquet(META_PATH, index=False)
print("Đã lưu metadata:", META_PATH)

dump(tfidf, f"{OUT_DIR}/tfidf_word12.joblib")
dump(svd,   f"{OUT_DIR}/svd256.joblib")


In [ ]:
import numpy as np, pandas as pd, os

OUT_DIR   = "social_networking/traditional/outputs_traditional"
EMB_PATH  = f"{OUT_DIR}/traditional_embeddings.npy"
META_PATH = f"{OUT_DIR}/metadata_traditional.parquet"
EVAL_PATH = "social_networking/traditional/evaluation_traditional.csv"


Xn   = np.load(EMB_PATH)              
meta = pd.read_parquet(META_PATH)     
assert Xn.shape[0] == len(meta)
print("Embeddings:", Xn.shape, "| Meta:", meta.shape)


In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Ks = [4, 8, 12, 16, 20, 24, 28, 30, 40, 50]

rows = []
for K in Ks:
    labels = KMeans(n_clusters=K, n_init="auto", random_state=42).fit_predict(Xn)
    sil = float(silhouette_score(Xn, labels, metric="cosine"))
    rows.append((K, sil))

df_k = pd.DataFrame(rows, columns=["K","silhouette_cosine"]).sort_values("silhouette_cosine", ascending=False)
best_K = int(df_k.iloc[0]["K"])
print(df_k.to_string(index=False))
print("best_K:", best_K)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
import pandas as pd, os, numpy as np

km = KMeans(n_clusters=best_K, n_init="auto", random_state=42).fit(Xn)
labels_km = km.labels_

uniq, cnt = np.unique(labels_km, return_counts=True)
print("best_K:", best_K, "| n_clusters:", len(uniq))
for k, c in zip(uniq, cnt):
    print(f"cluster {k}: {c}")

sil = float(silhouette_score(Xn, labels_km, metric="cosine"))
dbi = float(davies_bouldin_score(Xn, labels_km))
ch  = float(calinski_harabasz_score(Xn, labels_km))
print(f"[KMeans | Unsupervised] Silhouette={sil:.4f} | DBI={dbi:.4f} | CH={ch:.2f}")

y_true = LabelEncoder().fit_transform(meta["category_name"].astype(str).values)
def purity_score(y_true, y_pred):
    cm = contingency_matrix(y_true, y_pred)
    return cm.max(axis=0).sum() / cm.sum()
nmi = float(normalized_mutual_info_score(y_true, labels_km, average_method="arithmetic"))
ari = float(adjusted_rand_score(y_true, labels_km))
pur = float(purity_score(y_true, labels_km))
print(f"[KMeans | Supervised] NMI={nmi:.4f} | ARI={ari:.4f} | Purity={pur:.4f}")

rec_unsup = pd.DataFrame([{"model":"KMeans","metric":"cosine","K":best_K,"type":"unsupervised",
                           "silhouette_cosine":sil,"DBI":dbi,"CH":ch,"NMI":None,"ARI":None,"Purity":None}])
rec_sup   = pd.DataFrame([{"model":"KMeans","metric":"cosine","K":best_K,"type":"supervised",
                           "silhouette_cosine":None,"DBI":None,"CH":None,"NMI":nmi,"ARI":ari,"Purity":pur}])

df_out = pd.read_csv(EVAL_PATH) if os.path.exists(EVAL_PATH) else pd.DataFrame()
df_out = pd.concat([df_out, rec_unsup, rec_sup], ignore_index=True)
df_out.to_csv(EVAL_PATH, index=False)
print("Appended:", EVAL_PATH)


In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

EPS_LIST = [0.003, 0.005, 0.007, 0.009, 0.011]
MS_LIST  = [5, 8, 10, 12, 15]

rows = []
for eps in EPS_LIST:
    for ms in MS_LIST:
        labels = DBSCAN(eps=eps, min_samples=ms, metric="cosine", n_jobs=-1).fit_predict(Xn)
        mask = labels != -1
        if mask.sum() >= 2 and len(np.unique(labels[mask])) >= 2:
            sil = float(silhouette_score(Xn[mask], labels[mask], metric="cosine"))
        else:
            sil = np.nan
        rows.append((eps, ms, sil, int((~mask).sum()) / len(labels)))

df = pd.DataFrame(rows, columns=["eps","min_samples","silhouette_cosine","noise_frac"])
df_sorted = df.sort_values("silhouette_cosine", ascending=False, na_position="last")
print(df_sorted.head(15).to_string(index=False))

best = df_sorted.dropna(subset=["silhouette_cosine"]).iloc[0]
best_eps = float(best.eps)
best_ms  = int(best.min_samples)
print(f"\nChọn: eps={best_eps}, min_samples={best_ms} | sil={best.silhouette_cosine:.4f}")

labels_best = DBSCAN(eps=best_eps, min_samples=best_ms, metric="cosine", n_jobs=-1).fit_predict(Xn)
noise = int((labels_best == -1).sum())
uniq, cnt = np.unique(labels_best[labels_best != -1], return_counts=True)
print(f"clusters={len(uniq)} | noise={noise}/{len(labels_best)} ({noise/len(labels_best):.2%})")
for k, c in zip(uniq, cnt):
    print(f"cluster {k}: {c}")


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
import numpy as np, pandas as pd, os

db = DBSCAN(eps=best_eps, min_samples=best_ms, metric="cosine", n_jobs=-1).fit(Xn)
labels_db = db.labels_

noise = int((labels_db == -1).sum())
uniq, cnt = np.unique(labels_db[labels_db != -1], return_counts=True)
print(f"eps={best_eps}, min_samples={best_ms}")
print(f"clusters={len(uniq)} | noise={noise}/{len(labels_db)} ({noise/len(labels_db):.2%})")
for k, c in zip(uniq, cnt):
    print(f"cluster {k}: {c}")

mask = (labels_db != -1)
if mask.sum() >= 2 and len(np.unique(labels_db[mask])) >= 2:
    sil = float(silhouette_score(Xn[mask], labels_db[mask], metric="cosine"))
    dbi = float(davies_bouldin_score(Xn[mask], labels_db[mask]))
    ch  = float(calinski_harabasz_score(Xn[mask], labels_db[mask]))
else:
    sil = dbi = ch = float("nan")
print(f"[DBSCAN | Unsupervised] Silhouette={sil:.4f} | DBI={dbi} | CH={ch}")

y_true = LabelEncoder().fit_transform(meta["category_name"].astype(str).values)
def purity_score(y_true, y_pred):
    cm = contingency_matrix(y_true, y_pred)
    return cm.max(axis=0).sum() / cm.sum()
nmi = float(normalized_mutual_info_score(y_true, labels_db, average_method="arithmetic"))
ari = float(adjusted_rand_score(y_true, labels_db))
pur = float(purity_score(y_true, labels_db))
print(f"[DBSCAN | Supervised] NMI={nmi:.4f} | ARI={ari:.4f} | Purity={pur:.4f}")

rec_unsup = pd.DataFrame([{"model":"DBSCAN","metric":"cosine","K":"","type":"unsupervised",
                           "silhouette_cosine":sil,"DBI":dbi,"CH":ch,"NMI":None,"ARI":None,"Purity":None}])
rec_sup   = pd.DataFrame([{"model":"DBSCAN","metric":"cosine","K":"","type":"supervised",
                           "silhouette_cosine":None,"DBI":None,"CH":None,"NMI":nmi,"ARI":ari,"Purity":pur}])

df_out = pd.read_csv(EVAL_PATH) if os.path.exists(EVAL_PATH) else pd.DataFrame()
df_out = pd.concat([df_out, rec_unsup, rec_sup], ignore_index=True)
df_out.to_csv(EVAL_PATH, index=False)
print("Appended:", EVAL_PATH)


In [ ]:
import numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
import os

Ks = [4, 8, 12, 14, 16, 20, 24, 28, 30]
rows = []
for K in Ks:
    gmm = GaussianMixture(n_components=K, covariance_type="diag", reg_covar=1e-4, random_state=42)
    labels = gmm.fit_predict(Xn)
    sil = float(silhouette_score(Xn, labels, metric="cosine"))
    rows.append((K, sil))
df_k_gmm = pd.DataFrame(rows, columns=["K","silhouette_cosine"]).sort_values("silhouette_cosine", ascending=False)
best_K_gmm = int(df_k_gmm.iloc[0]["K"])
print(df_k_gmm.to_string(index=False))
print("best_K_gmm:", best_K_gmm)

gmm = GaussianMixture(n_components=best_K_gmm, covariance_type="diag", reg_covar=1e-4, random_state=42)
labels_gmm = gmm.fit_predict(Xn)
uniq, cnt = np.unique(labels_gmm, return_counts=True)
print("best_K_gmm:", best_K_gmm, "| n_clusters:", len(uniq))
for k, c in zip(uniq, cnt):
    print(f"cluster {k}: {c}")

sil = float(silhouette_score(Xn, labels_gmm, metric="cosine"))
dbi = float(davies_bouldin_score(Xn, labels_gmm))
ch  = float(calinski_harabasz_score(Xn, labels_gmm))
print(f"[GMM | Unsupervised] Silhouette={sil:.4f} | DBI={dbi:.4f} | CH={ch:.2f}")

y_true = LabelEncoder().fit_transform(meta["category_name"].astype(str).values)
def purity_score(y_true, y_pred):
    cm = contingency_matrix(y_true, y_pred)
    return cm.max(axis=0).sum() / cm.sum()
nmi = float(normalized_mutual_info_score(y_true, labels_gmm, average_method="arithmetic"))
ari = float(adjusted_rand_score(y_true, labels_gmm))
pur = float(purity_score(y_true, labels_gmm))
print(f"[GMM | Supervised] NMI={nmi:.4f} | ARI={ari:.4f} | Purity={pur:.4f}")

rec_unsup = pd.DataFrame([{"model":"GMM","metric":"cosine","K":best_K_gmm,"type":"unsupervised",
                           "silhouette_cosine":sil,"DBI":dbi,"CH":ch,"NMI":None,"ARI":None,"Purity":None}])
rec_sup   = pd.DataFrame([{"model":"GMM","metric":"cosine","K":best_K_gmm,"type":"supervised",
                           "silhouette_cosine":None,"DBI":None,"CH":None,"NMI":nmi,"ARI":ari,"Purity":pur}])

df_out = pd.read_csv(EVAL_PATH) if os.path.exists(EVAL_PATH) else pd.DataFrame()
df_out = pd.concat([df_out, rec_unsup, rec_sup], ignore_index=True)
df_out.to_csv(EVAL_PATH, index=False)
print("Appended:", EVAL_PATH)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

unsup_rows = [
    ("TF-IDF",        "KMeans",  0.2973, 2.5105, 1328.53),
    ("TF-IDF",        "DBSCAN",  0.9863, 0.1161, 16175.46),
    ("TF-IDF",        "GMM",     0.2763, 2.7836, 1275.06),
    ("PhoBERT",       "KMeans",  0.2047, 2.2994, 2418.46),
    ("PhoBERT",       "DBSCAN",  0.5558, 0.8371, 48.36),
    ("PhoBERT",       "GMM",     0.2637, 2.1819, 4331.95),
    ("E5-multilingual","KMeans", 0.1659, 2.8550, 1471.18),
    ("E5-multilingual","DBSCAN", 0.6701, 0.8439, 275.12),
    ("E5-multilingual","GMM",    0.1835, 3.0048, 1360.81),
]

df_unsup = pd.DataFrame(unsup_rows, columns=["Embedding","Algorithm","Silhouette","DBI","CH"])
df_unsup["Label"] = df_unsup["Embedding"] + " • " + df_unsup["Algorithm"]

def plot_barh_sorted(ax, df, value_col, title, ascending=False, xlabel=None):
    d = df.sort_values(value_col, ascending=ascending)
    ax.barh(d["Label"], d[value_col], color="#4C72B0")
    ax.set_title(title, fontsize=14, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=12)
    ax.tick_params(axis='y', labelsize=10)

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 16))
plt.subplots_adjust(hspace=0.5)

plot_barh_sorted(
    axes[0], df_unsup, "Silhouette",
    "Silhouette (cosine)",
    ascending=False, xlabel="Silhouette"
)
plot_barh_sorted(
    axes[1], df_unsup, "DBI",
    "Davies–Bouldin Index (DBI)",
    ascending=True, xlabel="DBI"
)
plot_barh_sorted(
    axes[2], df_unsup, "CH",
    "Calinski–Harabasz (CH)",
    ascending=False, xlabel="CH"
)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

sup_rows = [
    ("TF-IDF",         "KMeans",   0.9016, 0.8642, 0.9238),
    ("TF-IDF",         "DBSCAN",   0.0917, 0.0031, 0.1561),
    ("TF-IDF",         "GMM",      0.8353, 0.7479, 0.8617),
    ("PhoBERT",        "KMeans",   0.5477, 0.3781, 0.5329),
    ("PhoBERT",        "DBSCAN",   0.0058, 0.0006, 0.1107),
    ("PhoBERT",        "GMM",      0.2999, 0.1164, 0.2683),
    ("E5-multilingual","KMeans",   0.8515, 0.6364, 0.6579),
    ("E5-multilingual","DBSCAN",   0.7577, 0.5280, 1.0000),
    ("E5-multilingual","GMM",      0.9128, 0.8199, 0.7922),
]

df_sup = pd.DataFrame(sup_rows, columns=["Embedding","Algorithm","NMI","ARI","Purity"])
df_sup["Label"] = df_sup["Embedding"] + " • " + df_sup["Algorithm"]

def plot_barh_sorted(ax, df, value_col, title, xlabel=None):
    d = df.sort_values(value_col, ascending=False)
    ax.barh(d["Label"], d[value_col], color="#4C72B0")
    ax.set_title(title, fontsize=14, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=12)
    ax.tick_params(axis='y', labelsize=10)
    ax.set_xlim(0, 1.05)

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 16))
plt.subplots_adjust(hspace=0.5)

plot_barh_sorted(
    axes[0], df_sup, "NMI",
    "NMI — Supervised", xlabel="NMI"
)
plot_barh_sorted(
    axes[1], df_sup, "ARI",
    "ARI — Supervised", xlabel="ARI"
)
plot_barh_sorted(
    axes[2], df_sup, "Purity",
    "Purity — Supervised", xlabel="Purity"
)

plt.tight_layout()
plt.show()
